In [1]:
!pip install transformers sentencepiece safetensors


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import T5ForConditionalGeneration, T5Config, AutoTokenizer
from typing import Optional, Dict
from dataclasses import dataclass

@dataclass
class PointerGeneratorConfig:
    base_model_name: str = "t5-base"
    gate_hidden_size: int = 512
    max_input_length: int = 512
    max_target_length: int = 64
    copy_loss_weight: float = 0.0
    
class T5PointerGeneratorNetwork(nn.Module):
    """
    T5-based Pointer Generator Network.

    At each decoder step the model blends:
        final_dist = p_gen  * P_vocab(w)          (generate)
                   + (1-p_gen) * P_copy(w|source)  (copy)

    P_copy is obtained by scattering the cross-attention weights
    (which are defined over *source positions*) onto vocabulary indices
    using the actual input token IDs.
    """

    def __init__(self, pg_config: PointerGeneratorConfig):
        super().__init__()
        self.pg_config = pg_config

        # ── Load pretrained T5 ────────────────────────────────────────────
        self.t5 = T5ForConditionalGeneration.from_pretrained(pg_config.base_model_name)
        self.t5_config: T5Config = self.t5.config

        d_model = self.t5_config.d_model          # hidden size (768 for t5-base)
        d_embed = self.t5_config.d_model          # T5 uses tied embeddings
        vocab_size = self.t5_config.vocab_size    # 32128 for T5

        self.vocab_size = vocab_size

        # ── Pointer-generator gate ────────────────────────────────────────
        self.gate = PointerGeneratorGate(
            d_model=d_model,
            d_embed=d_embed,
            hidden_size=pg_config.gate_hidden_size,
        )

        # ── Projection: decoder hidden → vocabulary logits ─────────────────
        # Accessed via self.t5.lm_head directly in forward() — not aliased
        # here to avoid shared-tensor safetensors crash on checkpoint save.

        # ── Token embeddings (needed to get x_t for the gate) ─────────────
        # Accessed via self.t5.shared directly in forward() — same reason.

    # ─────────────────────────────────────────────────────────────────────────
    # Helper: Build the COPY distribution over the vocabulary
    # ─────────────────────────────────────────────────────────────────────────
    def _compute_copy_distribution(
        self,
        cross_attention_weights: torch.Tensor,  # (batch, n_heads, tgt_len, src_len)
        input_ids: torch.Tensor,                # (batch, src_len)
        attention_mask: torch.Tensor,           # (batch, src_len)  1=real token, 0=padding
        batch_size: int,
        tgt_len: int,
    ) -> torch.Tensor:
        """
        Convert position-level cross-attention → token-level copy probability.

        Steps:
          1. Average attention over all heads → (batch, tgt_len, src_len)
          2. Zero out attention on padding positions using attention_mask.
             Without this, floating-point residuals from T5's internal masking
             can scatter a small amount of probability mass onto the pad_token_id
             slot in the copy distribution, which is incorrect — padding tokens
             are not copyable source content.
          3. Renormalise so each row sums to 1. This is necessary here (and only
             here) because step 2 explicitly zeroes some positions, so the
             remaining weights no longer sum to exactly 1.0.
          4. Scatter-add onto vocabulary indices via input_ids.
        """
        # Average over attention heads
        # cross_attention_weights: (batch, heads, tgt_len, src_len)
        copy_attn = cross_attention_weights.float().mean(dim=1)  # (batch, tgt_len, src_len)

        # ── Zero attention mass on padding positions ───────────────────────
        # attention_mask: (batch, src_len) → expand to (batch, 1, src_len)
        # so it broadcasts across the tgt_len dimension.
        # Padding positions (mask == 0) are set to exactly 0.0 before scatter.
        pad_mask = attention_mask.unsqueeze(1).float()   # (batch, 1, src_len)
        copy_attn = copy_attn * pad_mask                 # (batch, tgt_len, src_len)

        # ── Renormalise ────────────────────────────────────────────────────
        # After zeroing padding positions the remaining weights may not sum to
        # 1.0 exactly. Divide each row by its sum to restore a valid probability
        # distribution before scattering onto vocabulary indices.
        # clamp(min=1e-9) guards against all-padding rows (should not occur in
        # practice but prevents a divide-by-zero if it ever does).
        copy_attn = copy_attn / copy_attn.sum(dim=-1, keepdim=True).clamp(min=1e-9)

        src_len = input_ids.size(1)

        # Initialise a zero tensor over the full vocabulary
        copy_dist = torch.zeros(
            batch_size, tgt_len, self.vocab_size,
            device=cross_attention_weights.device,
            dtype=torch.float32,
        )

        # Expand input_ids to align with tgt_len
        # expanded_input_ids: (batch, tgt_len, src_len)
        expanded_input_ids = input_ids.unsqueeze(1).expand(batch_size, tgt_len, src_len)

        # Scatter-add: for each (batch, tgt_pos, src_pos) triple,
        # add copy_attn[b, t, s] to copy_dist[b, t, input_ids[b, s]]
        copy_dist.scatter_add_(2, expanded_input_ids, copy_attn)

        return copy_dist  # (batch, tgt_len, vocab_size)

    # ─────────────────────────────────────────────────────────────────────────
    # Helper: Blend generate and copy distributions using p_gen
    # ─────────────────────────────────────────────────────────────────────────
    def _blend_distributions(
        self,
        vocab_logits: torch.Tensor,   # (batch, tgt_len, vocab_size)
        copy_dist: torch.Tensor,      # (batch, tgt_len, vocab_size)
        p_gen: torch.Tensor,          # (batch, tgt_len, 1)
    ) -> torch.Tensor:
        """
        Blend the vocabulary (generate) distribution and the copy distribution.

        final_dist = p_gen * P_vocab + (1 - p_gen) * P_copy

        We work in probability space (softmax) to allow meaningful blending,
        then return log-probabilities for use with NLLLoss.
        """
        vocab_prob = F.softmax(vocab_logits.float(), dim=-1)       # (batch, tgt_len, vocab_size)

        # Weighted sum of the two distributions
        final_dist = p_gen * vocab_prob + (1.0 - p_gen) * copy_dist  # (batch, tgt_len, vocab_size)

        # Clamp for numerical stability before taking log
        final_dist = final_dist.clamp(min=1e-9)

        return torch.log(final_dist)  # log-probs for NLLLoss

    # ─────────────────────────────────────────────────────────────────────────
    # Core forward pass
    # ─────────────────────────────────────────────────────────────────────────
    def forward(
        self,
        input_ids: torch.Tensor,                          # (batch, src_len)
        attention_mask: torch.Tensor,                     # (batch, src_len)
        labels: Optional[torch.Tensor] = None,            # (batch, tgt_len)
        decoder_input_ids: Optional[torch.Tensor] = None, # (batch, tgt_len)
        decoder_attention_mask: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Full forward pass with pointer-generator blending.

        Returns a dict with:
            loss          : combined generation + copy loss (if labels given)
            logits        : blended log-prob distribution (batch, tgt_len, vocab_size)
            p_gen         : gate values (batch, tgt_len, 1)
        """
        batch_size = input_ids.size(0)

        # ── 1. Prepare decoder inputs ──────────────────────────────────────
        # During training, labels are shifted right to form decoder inputs.
        # During inference, we provide decoder_input_ids manually.
        if decoder_input_ids is None and labels is not None:
            # Labels contain -100 at padding positions (the ignore index for loss).
            # _shift_right does not strip these — it would pass -100 as real token
            # IDs into the decoder embedding table, causing an index-out-of-range
            # error or silent corruption. Replace -100 with pad_token_id first.
            shifted_labels = labels.clone()
            shifted_labels[shifted_labels == -100] = self.t5.config.pad_token_id
            decoder_input_ids = self.t5._shift_right(shifted_labels)

        # ── 2. Run T5 encoder + decoder ───────────────────────────────────
        # output_attentions=True so we can harvest cross-attention for copying
        t5_outputs = self.t5(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
        )

        # ── 3. Extract hidden states ───────────────────────────────────────
        # decoder_hidden_states is a tuple of (n_layers+1) tensors,
        # each of shape (batch, tgt_len, d_model).
        # We take the LAST layer's hidden state.
        decoder_hidden = t5_outputs.decoder_hidden_states[-1]  # (batch, tgt_len, d_model)
        tgt_len = decoder_hidden.size(1)

        # ── 4. Build context vector from cross-attention ───────────────────
        # cross_attentions is a tuple of (n_layers) tensors,
        # each of shape (batch, heads, tgt_len, src_len).
        # We use the LAST decoder layer's cross-attention.
        last_cross_attn = t5_outputs.cross_attentions[-1]  # (batch, heads, tgt_len, src_len)

        # Average over heads → (batch, tgt_len, src_len)
        avg_cross_attn = last_cross_attn.mean(dim=1)

        # Encoder hidden states: last layer → (batch, src_len, d_model)
        encoder_hidden = t5_outputs.encoder_last_hidden_state

        # Context vector: weighted sum of encoder states
        # (batch, tgt_len, src_len) x (batch, src_len, d_model) → (batch, tgt_len, d_model)
        context_vector = torch.bmm(avg_cross_attn, encoder_hidden)

        # ── 5. Token embeddings for the gate's x_t signal ─────────────────
        # x_t = embedding of the token fed into the decoder at each step
        token_embedding = self.t5.shared(decoder_input_ids)  # (batch, tgt_len, d_embed)

        # ── 6. Compute p_gen ──────────────────────────────────────────────
        p_gen = self.gate(
            decoder_hidden=decoder_hidden,    # (batch, tgt_len, d_model)
            context_vector=context_vector,    # (batch, tgt_len, d_model)
            token_embedding=token_embedding,  # (batch, tgt_len, d_embed)
        )  # → (batch, tgt_len, 1)

        # ── 7. Vocabulary (generate) distribution ─────────────────────────
        # t5_outputs.logits shape: (batch, tgt_len, vocab_size)
        vocab_logits = t5_outputs.logits

        # ── 8. Copy distribution ──────────────────────────────────────────
        copy_dist = self._compute_copy_distribution(
            cross_attention_weights=last_cross_attn,
            input_ids=input_ids,
            attention_mask=attention_mask,
            batch_size=batch_size,
            tgt_len=tgt_len,
        )  # (batch, tgt_len, vocab_size)

        # ── 9. Blend distributions ─────────────────────────────────────────
        blended_log_probs = self._blend_distributions(
            vocab_logits=vocab_logits,
            copy_dist=copy_dist,
            p_gen=p_gen,
        )  # (batch, tgt_len, vocab_size)  — log-probs

        # ── 10. Compute losses ─────────────────────────────────────────────
        loss = None
        if labels is not None:
            # Replace padding token (-100) positions before computing loss
            # NLLLoss ignores index -100 by default
            loss_fct = nn.NLLLoss(ignore_index=-100)

            # Reshape for NLLLoss: expects (N, C) and (N,)
            generation_loss = loss_fct(
                blended_log_probs.view(-1, self.vocab_size),
                labels.view(-1),
            )

            loss = generation_loss

            # ── Optional copy auxiliary loss ───────────────────────────────
            # Encourages the model to copy when the target token appears
            # in the source, making the pointer more reliable.
            if self.pg_config.copy_loss_weight > 0.0:
                copy_loss = self._compute_copy_auxiliary_loss(
                    labels=labels,
                    input_ids=input_ids,
                    p_gen=p_gen,
                )
                loss = loss + self.pg_config.copy_loss_weight * copy_loss

        return {
            "loss": loss,
            "logits": blended_log_probs,   # (batch, tgt_len, vocab_size)
            "p_gen": p_gen,                # (batch, tgt_len, 1) — for analysis
        }

    # ─────────────────────────────────────────────────────────────────────────
    # Auxiliary copy loss
    # ─────────────────────────────────────────────────────────────────────────
    def _compute_copy_auxiliary_loss(
        self,
        labels: torch.Tensor,     # (batch, tgt_len)
        input_ids: torch.Tensor,  # (batch, src_len)
        p_gen: torch.Tensor,      # (batch, tgt_len, 1)
    ) -> torch.Tensor:
        """
        Auxiliary loss that pushes p_gen toward 0 (copy) when the
        gold target token is present in the source sequence.

        This provides extra gradient signal to the gate when copying
        is the right action.
        """
        # For each target token, check if it appears anywhere in the source
        # labels:    (batch, tgt_len)
        # input_ids: (batch, src_len)

        # Broadcast comparison: (batch, tgt_len, 1) vs (batch, 1, src_len)
        is_copyable = (labels.unsqueeze(2) == input_ids.unsqueeze(1)).any(dim=2)  # (batch, tgt_len)

        # Ignore padding positions in labels
        valid_mask = (labels != -100).float()   # (batch, tgt_len)
        is_copyable_float = is_copyable.float() * valid_mask  # (batch, tgt_len)

        # When is_copyable=1: we want p_gen → 0 → minimise p_gen
        # When is_copyable=0: no constraint from copy loss
        p_gen_squeezed = p_gen.squeeze(-1)       # (batch, tgt_len)

        # MSE toward 0 where copyable, 0 elsewhere
        copy_loss = (is_copyable_float * p_gen_squeezed ** 2).sum()
        n_valid = valid_mask.sum().clamp(min=1)

        return copy_loss / n_valid

C:\Users\Ajith Kumar KP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class PointerGeneratorGate(nn.Module):
    """
    Two-layer MLP that produces p_gen ∈ (0, 1).

    p_gen = 1  → generate from the vocabulary distribution
    p_gen = 0  → copy from the source (attention distribution over input tokens)
    """

    def __init__(self, d_model: int, d_embed: int, hidden_size: int):
        """
        Args:
            d_model     : T5 hidden dimension (e.g. 768 for t5-base)
            d_embed     : T5 token embedding dimension (same as d_model in T5)
            hidden_size : intermediate projection size for the gate MLP
        """
        super().__init__()

        # Input to the gate: concatenation of three vectors
        #   [decoder_hidden (d_model), context_vector (d_model), token_embedding (d_embed)]
        gate_input_size = d_model + d_model + d_embed

        self.gate_network = nn.Sequential(
            nn.Linear(gate_input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 1),   # scalar output
            nn.Sigmoid()                 # squash to (0, 1)
        )

    def forward(
        self,
        decoder_hidden: torch.Tensor,   # (batch, 1, d_model)
        context_vector: torch.Tensor,   # (batch, 1, d_model)
        token_embedding: torch.Tensor,  # (batch, 1, d_embed)
    ) -> torch.Tensor:
        """
        Returns:
            p_gen : (batch, 1, 1) – generation probability at this timestep
        """
        # Concatenate along the last (feature) dimension
        gate_input = torch.cat([decoder_hidden, context_vector, token_embedding], dim=-1)
        p_gen = self.gate_network(gate_input)   # (batch, 1, 1)
        return p_gen

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import T5ForConditionalGeneration, T5Config, AutoTokenizer
from typing import Optional, Dict
from dataclasses import dataclass

# -----------------------------
# CONFIG
# -----------------------------


# -----------------------------
# LOAD MODEL
# -----------------------------

 # use your PGN class

checkpoint_path = "D:/phase1_squad/phase1_squad/checkpoint-8142"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("t5-base")

config = PointerGeneratorConfig()

model = T5PointerGeneratorNetwork(config)

from safetensors.torch import load_file

state_dict = load_file(f"{checkpoint_path}/model.safetensors")

# Rename keys so they match the PGN wrapper structure
new_state_dict = {}

for k, v in state_dict.items():
    if k.startswith("encoder.") or k.startswith("decoder.") or k.startswith("shared.") or k.startswith("lm_head"):
        new_state_dict["t5." + k] = v
    else:
        new_state_dict[k] = v

model.load_state_dict(new_state_dict, strict=False)

model.to(device)
model.eval()


# -----------------------------
# GREEDY DECODING
# -----------------------------

def generate_answer(question, context):

    input_text = f"question: {question} context: {context}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=config.max_input_length
    ).to(device)

    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]], device=device)

    generated_tokens = []

    with torch.no_grad():

        for _ in range(config.max_target_length):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids
            )

            final_dist = torch.exp(outputs["logits"][:, -1, :])

            next_token = torch.argmax(final_dist, dim=-1)

            token_id = next_token.item()

            if token_id == tokenizer.eos_token_id:
                break

            generated_tokens.append(token_id)

            decoder_input_ids = torch.cat(
                [decoder_input_ids, next_token.unsqueeze(0)],
                dim=-1
            )

    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer


# -----------------------------
# TEST EXAMPLE
# -----------------------------

question = "What is London?"

context = """
Paris is the capital of France. It is a very big city. Too much gay people.
"""

answer = generate_answer(question, context)

print("\nQuestion:", question)
print("Answer:", answer)


Question: What is London?
Answer: a very big city 
